# 50 — M4 forward derivatives

M4 専用ノートブックです。現時点では M3 の実環境受理を検証する fail-closed ゲートだけを収録し、微分や influence bound を実装済みとして扱いません。

In [ ]:
# 必ず最初に pytest を用意する。
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)


In [ ]:
from pathlib import Path
import hashlib, json, os

PREVIOUS, CURRENT, RUN_ENV = 'M3', 'M4', 'VALIDATED_RG_M3_RUN_ID'
run_id = os.environ.get(RUN_ENV)
if not run_id:
    raise RuntimeError(f'{RUN_ENV} を受理済み M3 run ID に設定してください。M4 はまだ開始されません。')
if Path(run_id).name != run_id or not run_id.startswith('M3-'):
    raise RuntimeError('M3 run ID の形式が不正です。')
persist_root = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
acceptance_path = persist_root / 'runs' / run_id / 'reports' / 'M3_acceptance.json'
if not acceptance_path.is_file():
    raise RuntimeError(f'M3 acceptance artifact がありません: {acceptance_path}')
acceptance = json.loads(acceptance_path.read_text(encoding='utf-8'))
expected = {'milestone': 'M3', 'phase': 'M3_COMPLETE', 'status': 'PASS', 'certification_status': 'NOT_CERTIFIED'}
if any(acceptance.get(k) != v for k, v in expected.items()):
    raise RuntimeError('M3 acceptance artifact の identity/status が不正です。')
STAGE_GATE = {
    'current_milestone': CURRENT, 'predecessor': PREVIOUS, 'status': 'PREDECESSOR_ACCEPTED',
    'acceptance_path': str(acceptance_path),
    'acceptance_sha256': hashlib.sha256(acceptance_path.read_bytes()).hexdigest(),
    'implementation_status': 'LOCKED_PENDING_M4_IMPLEMENTATION_REVIEW',
    'certification_status': 'NOT_CERTIFIED',
}
STAGE_GATE


## M4 実装境界

直前セルが通っても M4 完了ではありません。forward AD/JVP、finite-difference cross-check、微分 truncation error、influence accounting、M4 acceptance report は未実装です。M3 の人間レビュー後に、このノートブックだけを拡張します。